# Exercise 4.3: training a postprocessing network
In this exercise, we will train a model for a postprocessing task as described in Section 5. 

Follow these steps for each inverse problem:

1. First, we implement an ImageDataset child class to generate ellipses and rectangles.
2. Define the forward tomographic operator with angles in $[0, \pi/2]$.
3. Define the reconstruction architecture. The postprocessing architecture is given by $\mathcal R_\theta = \Phi_\theta \circ A^\dagger$, which can be realized in DeepInverse via [ArtifactRemoval model](https://deepinv.github.io/deepinv/api/stubs/deepinv.models.ArtifactRemoval.html). For $\Phi_\theta$ you can use [U-Net](https://arxiv.org/pdf/1505.04597) or a simpler [DnCNN](http://deepinv.github.io/deepinv/api/stubs/deepinv.models.DnCNN.html).
5. Train the reconstruction network on ellipses. At each training epoch, plot the PSNR of reconstructions and example images of source, measurements, and reconstruction.
6. Test the trained network on rectangles and compare the PSNR before and after training on the testset.

## Change runtime type and install DeepInverse package
First, we install the deepinverse library. Remeber to change the runtime of this notebook select ``Runtime`` -> ``Change runtime type`` and pick an available hardware accelerator.

In [ ]:
!pip install git+https://github.com/deepinv/deepinv.git#egg=deepinv

In [ ]:
import deepinv as dinv
from torch.utils.data import DataLoader
import torch
from pathlib import Path
from torchvision import transforms
from deepinv.utils.demo import load_dataset

import matplotlib.pyplot as plt

# Setup paths for data loading and checkpoints

In [ ]:
BASE_DIR = Path(".")
DATA_DIR = BASE_DIR / "measurements"
CKPT_DIR = BASE_DIR / "ckpts"

# Set the global random seed from pytorch to ensure reproducibility of the example.
torch.manual_seed(0)

device = dinv.utils.get_freer_gpu() if torch.cuda.is_available() else "cpu"

# Generate dataset
We create a simple dataset class for generating images of ellipses and rectangles. As a basis, we use [PlaceholderDataset](https://deepinv.github.io/deepinv/_modules/deepinv/datasets/utils.html#PlaceholderDataset) class. 
Use `torch.randint` (https://docs.pytorch.org/docs/stable/generated/torch.randint.html) to generate random points and define random shapes.

In [ ]:
class ShapeDataset(dinv.datasets.ImageDataset):
    shape_ellipse: bool = True

    def __init__(self, n=1, shape=(1, 64, 64), shape_ellipse = True):
        self.n = n # number of images in the dataset
        self.shape = shape # shape of images in order (number of channels, width, height)
        self.shape_ellipse = shape_ellipse # ellipse or rectangle?

    def __len__(self):
        return self.n


    def __getitem__(self, index):
        # This function should return 1 tensor of size self.shape, either a rectangle or an ellipse
        nx = self.shape[1]
        ny = self.shape[2]

        # Create index grid using torch.arange and torch.meshgrid
        xgrid = torch.arange(nx)
        ygrid = torch.arange(ny)
        grid_x, grid_y = torch.meshgrid(xgrid, ygrid, indexing='ij')
                
        if self.shape_ellipse:
            # Generate a random ellipse
            # Recall that an ellipse is given by the equation (x - x_0)^2 / a^2 + (y - y_0)^2/b^2 <= 1,
            # where (x_0, y_0) is the center and (a,b) define the length of the sides 
            
            # Center point
            c = FILL_THIS_PART
            
            # Length of sides
            s = FILL_THIS_PART
            
            x = FILL_THIS_PART
            return x[None,].float()
        else:
            # Generate a random rectangle
            # Recall that rectangle is defined by x_0 <= x <= x_1 and y_0 <= y <= y_1,
            # where (x_0, y_0) is its bottom left point and (x_1, y_1) is its top right point
            
            x1 = FILL_THIS_PART
            x2 = FILL_THIS_PART

            # Why do we need this?
            left= x1.minimum(x2)
            right = x1.maximum(x2)
            
            c1 = FILL_THIS_PART # Check x conditions (see torch.logical_and)
            
            y1 = FILL_THIS_PART
            y2 = FILL_THIS_PART
            
            bottom= y1.minimum(y2)
            top = y1.maximum(y2)
            
            c2 = FILL_THIS_PART # Check y conditions 
            c3 = c2.logical_and(c1)
            return c3[None,].float()

In [ ]:
operation = "tomography"
train_dataset_name = "ellipses"
test_dataset_name = "square"
img_size = 128 if torch.cuda.is_available() else 32

n_images_train = 128 
n_images_test = 32

# Why do we choose shape as (1,img_size,img_size)?
train_dataset = ShapeDataset(n=n_images_train, shape=(1,img_size, img_size), shape_ellipse = True)
val_dataset = ShapeDataset(n=n_images_test, shape=(1,img_size, img_size), shape_ellipse = True)
test_dataset = ShapeDataset(n=n_images_test, shape=(1,img_size, img_size), shape_ellipse = False)

In [ ]:
# Plot examples of the generated images 

# From the train set
x_tr = train_dataset[0]
plt.imshow(x_tr[0])
plt.show()

# From the test set
x_test = test_dataset[0]
plt.imshow(x_test[0])
plt.show()

# Define forward operator and prepare dataloaders
Define the forward tomographic operator with 90 equally spaced angles in $[0, \pi/2]$. See `torch.linspace` (https://docs.pytorch.org/docs/stable/generated/torch.linspace.html#torch-linspace) for details. 

In [ ]:
n_channels = 3  # 3 for color images, 1 for gray-scale images
probability_mask = 0.5  # probability to mask pixel

# Generate inpainting operator
physics = dinv.physics.Tomography(
    img_width=img_size,
    angles=torch.linspace(FILL_THIS_PART).to(device), # in degrees
    normalize=False,
    device=device,
    # noise_model=dinv.physics.GaussianNoise(sigma=noise_level_img), # optional complication
)

# For some problems, it is more convenient to pregenerate the measurements.
# Later, we can set online_measurements = False for Trainer
# It can be done via generate_dataset function
# n_images_max = (
#     1000 if torch.cuda.is_available() else 50
# )  # maximal number of images used for training

# my_dataset_name = "demo_training_tomograpy"
# measurement_dir = DATA_DIR / train_dataset_name / operation
# deepinv_datasets_path = dinv.datasets.generate_dataset(
#     train_dataset=train_dataset,
#     test_dataset=test_dataset,
#     physics=physics,
#     device=device,
#     save_dir=measurement_dir,
#     train_datapoints=n_images_max,
#     num_workers=num_workers,
#     dataset_filename=str(my_dataset_name),
# )

# train_dataset = dinv.datasets.HDF5Dataset(path=deepinv_datasets_path, train=True)
# test_dataset = dinv.datasets.HDF5Dataset(path=deepinv_datasets_path, train=False)

# deepinv_datasets_path_val = dinv.datasets.generate_dataset(
#     train_dataset=val_dataset,
#     test_dataset=None,
#     physics=physics,
#     device=device,
#     save_dir=measurement_dir,
#     train_datapoints=n_images_max,
#     num_workers=num_workers,
#     dataset_filename=str(my_dataset_name),
# )
# val_dataset = dinv.datasets.HDF5Dataset(path=deepinv_datasets_path_val, train=True)

num_workers = 4 if torch.cuda.is_available() else 1

train_batch_size = 16 if torch.cuda.is_available() else 1
test_batch_size = 16 if torch.cuda.is_available() else 1

# Set up dataloaders. Do we need to shuffle in this setup?

train_dataloader = FILL_THIS_PART

val_dataloader = FILL_THIS_PART

test_dataloader = FILL_THIS_PART

# Set up the reconstruction network

Set up reconstruction model $R_\theta = \Phi_\theta \circ A^\dagger$ based on [ArtifactRemoval model](https://deepinv.github.io/deepinv/api/stubs/deepinv.models.ArtifactRemoval.html).
Note we can pass the argument ``mode=direct``, ``mode=adjoint`` or ``mode=pinv``. Which one should be used here?

In [ ]:
# choose backbone model
backbone = dinv.models.UNet(
    in_channels=1, out_channels=1, scales=3, batch_norm=False
).to(device)

# choose a reconstruction architecture
model = dinv.models.ArtifactRemoval(backbone, mode=FILL_THIS_PART)

print(model)

# Train the model
We train the model using the [deepinv.Trainer](https://deepinv.github.io/deepinv/api/stubs/deepinv.Trainer.html#deepinv.Trainer) class.

We perform supervised learning and use the mean squared error as loss function. This can be easily done using the [deepinv.loss.SupLoss](https://deepinv.github.io/deepinv/api/stubs/deepinv.loss.SupLoss.html#deepinv.loss.SupLoss) class.


In [ ]:
trainer = FILL_THIS_PART

In [ ]:
trainer.train()

# Test the network
We can now test the trained network using the [deepinv.test()](https://deepinv.github.io/deepinv/api/stubs/deepinv.test.html#deepinv.test) function.

The testing function will compute test_psnr metrics and plot and save the results.

In [ ]:
# load model with best validation PSNR
model = trainer.load_best_model()

trainer.plot_images = True
trainer.test(val_dataloader)

In [ ]:
trainer.test(test_dataloader)

# Test reconstruction on Shepp-Logan phantom

In [ ]:
x = dinv.utils.load_example(
    "SheppLogan.png",
    img_size=img_size,
    grayscale=True,
    resize_mode="resize",
    device=device,
)

y = physics(x)

y_hat = model(y,physics)

plt.subplots(1, 3, figsize=(10, 5))
plt.subplot(1, 3, 1)
plt.imshow(x[0,0].cpu())
plt.axis("off")
plt.title('Source')

plt.subplot(1, 3, 2)
plt.imshow(y[0,0].cpu())
plt.axis("off")
plt.title('Measurements')

print(y_hat.shape)

plt.subplot(1, 3, 3)
plt.imshow(y_hat[0,0].detach().cpu())
plt.axis("off")
plt.title('Reconstruction')

plt.show()